# Segmento 2: Qdrant Vector Database

Abbiamo visto la cosine similarity a mano. Ora usiamo un vero vector database, **Qdrant**, che gestisce storage, indicizzazione e query veloci per noi.

Assicuriamoci che Qdrant sia attivo:
```bash
docker compose up -d
```

Dashboard: http://localhost:6333/dashboard

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from datasets import load_dataset
import numpy as np
import pandas as pd
import ast, os

load_dotenv()
openai_client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)
EMBED_MODEL = "text-embedding-3-small"
COLLECTION = "recipes"

# Carichiamo il dataset (stesso sottoinsieme del segmento 1)
ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
df = ds.to_pandas().drop(columns=["Unnamed: 0", "Image_Name"])
df = df.sample(n=1000, random_state=42).reset_index(drop=True)
print(f"{len(df)} ricette caricate")
print(f"Stato Qdrant: {qdrant.get_collections()}")

/home/federicor/aidea/boolean/boolean-master-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1000 ricette caricate
Stato Qdrant: collections=[]


## Creare la collection

Una **collection** in Qdrant è come una tabella: contiene vettori di dimensione fissa con una specifica metrica di distanza.

In [14]:
# Cancelliamo se esiste già (per poter rieseguire)
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=1536,  # dimensione di text-embedding-3-small
        distance=Distance.COSINE,
    ),
)
print(f"Collection '{COLLECTION}' creata")

Collection 'recipes' creata


## Generare gli embeddings e inserirli in Qdrant

Per ogni ricetta:
1. Concateniamo titolo + ingredienti + istruzioni
2. Generiamo l'embedding con OpenAI
3. Salviamo il vettore + metadati della ricetta (titolo, ingredienti) come **punto** in Qdrant

In [15]:
def safe_str(val):
    """Convert to string, handling NaN/None."""
    if pd.isna(val):
        return ""
    return str(val)

def recipe_to_text(row):
    title = safe_str(row["Title"])
    ingredients = safe_str(row["Ingredients"])
    instructions = safe_str(row["Instructions"])
    return f"Title: {title}\nIngredients: {ingredients}\nInstructions: {instructions}"

def get_embeddings(texts):
    response = openai_client.embeddings.create(input=texts, model=EMBED_MODEL)
    return [item.embedding for item in response.data]

# Prepariamo i testi
df["full_text"] = df.apply(recipe_to_text, axis=1)

# Embeddiamo e inseriamo in batch
batch_size = 100

for i in range(0, len(df), batch_size):
    batch_df = df.iloc[i:i+batch_size]
    texts = batch_df["full_text"].tolist()
    embeddings = get_embeddings(texts)
    
    points = [
        PointStruct(
            id=i + j,
            vector=embeddings[j],
            payload={
                "title": safe_str(batch_df.iloc[j]["Title"]),
                "ingredients": safe_str(batch_df.iloc[j]["Ingredients"]),
                "instructions": safe_str(batch_df.iloc[j]["Instructions"])[:500],
            },
        )
        for j in range(len(batch_df))
    ]
    
    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"Inserite {min(i+batch_size, len(df))}/{len(df)} ricette")

print(f"\nFatto! Info collection:")
print(qdrant.get_collection(COLLECTION))

Inserite 100/1000 ricette
Inserite 200/1000 ricette
Inserite 300/1000 ricette
Inserite 400/1000 ricette
Inserite 500/1000 ricette
Inserite 600/1000 ricette
Inserite 700/1000 ricette
Inserite 800/1000 ricette
Inserite 900/1000 ricette
Inserite 1000/1000 ricette

Fatto! Info collection:
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=1000 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=Non

## Interrogare Qdrant

Ora possiamo cercare per significato, la stessa cosa che facevamo con NumPy, ma Qdrant gestisce indicizzazione e calcolo delle distanze.

**Andate a vedere la dashboard**: http://localhost:6333/dashboard, vedrete la collection e i punti.

In [16]:
def search_qdrant(query, top_n=5):
    """Search recipes in Qdrant by semantic similarity."""
    query_embedding = get_embeddings([query])[0]
    
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_embedding,
        limit=top_n,
        with_payload=True,
    )
    return results.points

# Proviamo
print("Ricerca: 'tomato and basil dish'\n")
for point in search_qdrant("tomato and basil dish"):
    print(f"  {point.score:.4f}  {point.payload['title']}")

Ricerca: 'tomato and basil dish'

  0.5940  Baked Garden Tomatoes with Cheese
  0.5828  Cherry Tomato Polenta Tartlets with Basil Mayonnaise
  0.5753  Smoked Summer Tomato Basil Butter
  0.5634  Cherry Tomatoes Stuffed with Marinated Feta
  0.5471  Classic Tomato Sauce


In [17]:
print("Ricerca: 'quick weeknight dinner'\n")
for point in search_qdrant("quick weeknight dinner"):
    print(f"  {point.score:.4f}  {point.payload['title']}")

Ricerca: 'quick weeknight dinner'

  0.4669  Pasta With 15-Minute Meat Sauce
  0.4578  Mac 'n' Cheese Minis
  0.4412  Quick Coq au Vin
  0.4384  Seafood Stew for Two
  0.4371  Creamy Farfalle with Salmon and Peas


In [ ]:
print("Ricerca: 'recipe with lemon juice'\n")
for point in search_qdrant("recipe with lemon juice"):
    print(f"  {point.score:.4f}  {point.payload['title']}")

In [18]:
# Bonus: una query che mostra la comprensione dell'intento
print("Ricerca: 'something warm and comforting for a cold day'\n")
for point in search_qdrant("something warm and comforting for a cold day"):
    print(f"  {point.score:.4f}  {point.payload['title']}")

Ricerca: 'something warm and comforting for a cold day'

  0.3548  Hot Toddy
  0.3261  Chilled Tomato and Stone Fruit Soup
  0.3219  Winter Minestrone
  0.3174  Warm-Spiced Saucy Lamb Stew
  0.3152  Chilled Watercress-Spinach Soup


## Cosa abbiamo costruito

Ora abbiamo un **vector database persistente e indicizzato** con 1.000 ricette che possiamo interrogare per significato.

Prossimo passo: trasformiamo questa ricerca in un tool per un LLM.